# 102: 固定長チャンク x オーバーラップスイープ

Notebook 101 で準備した記事データを使い、固定長トークンチャンクの **chunk_size** と **overlap** の
全組み合わせで embedding を生成し、検索品質指標 (Recall@10, MRR, ADS) を評価する。

**モデル**:
- `intfloat/multilingual-e5-base` (768d) — chunk_size: [128, 256, 384, 512], overlap: [0, 32, 64, 128]
- `Qwen/Qwen3-Embedding-0.6B` (1024d) — chunk_size: [256, 512, 1024, 2048], overlap: [0, 64, 128, 256]

**制約**: overlap < chunk_size / 2

**出力**: ヒートマップによる最適オーバーラップ率の可視化、収穫逓減点の特定

In [1]:
import gc, os, pickle, re, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer
from tqdm.auto import tqdm
sys.path.insert(0, "../src")

SEED = 42
DATA_DIR = "../data"

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def chunk_fixed_token(text, tokenizer, size, overlap=0):
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    if len(token_ids) == 0:
        return []
    step = max(1, size - overlap)
    chunks = []
    for start in range(0, len(token_ids), step):
        chunk_ids = token_ids[start : start + size]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True).strip()
        if chunk_text:
            chunks.append(chunk_text)
        if start + size >= len(token_ids):
            break
    return chunks

def compute_retrieval_metrics(embeddings, article_ids, k=10, n_queries=1000):
    n = len(embeddings)
    rng = np.random.default_rng(SEED)
    query_ids = rng.choice(n, min(n_queries, n), replace=False)
    recalls, mrrs = [], []
    for qid in query_ids:
        same_art = np.where((article_ids == article_ids[qid]) & (np.arange(n) != qid))[0]
        if len(same_art) == 0:
            continue
        sims = embeddings[qid] @ embeddings.T
        sims[qid] = -2.0
        topk_ids = np.argsort(sims)[-k:][::-1]
        found = len(set(same_art) & set(topk_ids))
        recalls.append(found / min(len(same_art), k))
        rr = 0.0
        for rank, idx in enumerate(topk_ids, 1):
            if idx in same_art:
                rr = 1.0 / rank
                break
        mrrs.append(rr)
    return {f"Recall@{k}": float(np.mean(recalls)), "MRR": float(np.mean(mrrs)), "n_queries": len(recalls)}

def compute_ads(embeddings, article_ids, n_sample_pairs=50000):
    n = len(embeddings)
    rng = np.random.default_rng(SEED)
    idx1 = rng.choice(n, n_sample_pairs, replace=True)
    idx2 = rng.choice(n, n_sample_pairs, replace=True)
    mask = idx1 != idx2
    idx1, idx2 = idx1[mask], idx2[mask]
    cos_sims = np.sum(embeddings[idx1] * embeddings[idx2], axis=1)
    same_article = (article_ids[idx1] == article_ids[idx2]).astype(int)
    ads = roc_auc_score(same_article, cos_sims)
    intra_mask = same_article == 1
    inter_mask = same_article == 0
    intra_mean = float(np.mean(cos_sims[intra_mask])) if intra_mask.sum() > 0 else 0.0
    inter_mean = float(np.mean(cos_sims[inter_mask])) if inter_mask.sum() > 0 else 0.0
    return {"ADS (AUC)": ads, "IIG": intra_mean - inter_mean}

print("Setup complete.")

Setup complete.


In [2]:
with open(os.path.join(DATA_DIR, "101_articles_ja.pkl"), "rb") as f:
    articles_ja = pickle.load(f)
with open(os.path.join(DATA_DIR, "101_articles_en.pkl"), "rb") as f:
    articles_en = pickle.load(f)

tokenizer_e5 = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
tokenizer_qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B")

print(f"JA: {len(articles_ja)} articles, EN: {len(articles_en)} articles")

JA: 1000 articles, EN: 1000 articles


## 2. パラメータグリッドの定義

In [3]:
# E5-base: shorter chunks (max 512 tokens)
e5_grid = []
for size in [128, 256, 384, 512]:
    for overlap in [0, 32, 64, 128]:
        if overlap < size / 2:  # overlap must be < 50% of size
            e5_grid.append((size, overlap))

# Qwen3: longer chunks
qwen_grid = []
for size in [256, 512, 1024, 2048]:
    for overlap in [0, 64, 128, 256]:
        if overlap < size / 2:
            qwen_grid.append((size, overlap))

print(f"E5 grid: {len(e5_grid)} combos")
for s, o in e5_grid:
    print(f"  size={s}, overlap={o}")
print(f"\nQwen3 grid: {len(qwen_grid)} combos")
for s, o in qwen_grid:
    print(f"  size={s}, overlap={o}")

E5 grid: 13 combos
  size=128, overlap=0
  size=128, overlap=32
  size=256, overlap=0
  size=256, overlap=32
  size=256, overlap=64
  size=384, overlap=0
  size=384, overlap=32
  size=384, overlap=64
  size=384, overlap=128
  size=512, overlap=0
  size=512, overlap=32
  size=512, overlap=64
  size=512, overlap=128

Qwen3 grid: 13 combos
  size=256, overlap=0
  size=256, overlap=64
  size=512, overlap=0
  size=512, overlap=64
  size=512, overlap=128
  size=1024, overlap=0
  size=1024, overlap=64
  size=1024, overlap=128
  size=1024, overlap=256
  size=2048, overlap=0
  size=2048, overlap=64
  size=2048, overlap=128
  size=2048, overlap=256


## 3. チャンキング + Embedding スイープ

In [ ]:
def chunk_articles(articles, tokenizer, size, overlap):
    """全記事をチャンク分割し、チャンクテキストとarticle_idsを返す"""
    all_chunks = []
    all_article_ids = []
    for art in articles:
        chunks = chunk_fixed_token(art["text"], tokenizer, size, overlap)
        if len(chunks) >= 2:  # 最低2チャンク必要（検索評価のため）
            all_chunks.extend(chunks)
            all_article_ids.extend([art["article_id"]] * len(chunks))
    return all_chunks, np.array(all_article_ids)


def adaptive_batch_size(chunk_size: int, base_batch: int = 64) -> int:
    """チャンクのトークン数に応じてbatch_sizeを調整する。
    長いシーケンスほどattention行列が大きくなるため小さくする。"""
    if chunk_size <= 256:
        return base_batch
    elif chunk_size <= 512:
        return max(8, base_batch // 2)
    elif chunk_size <= 1024:
        return max(4, base_batch // 4)
    else:  # 2048+
        return max(2, base_batch // 8)


def run_sweep(model_id, prefix, tokenizer, grid, articles_dict, batch_size=64):
    """1モデルの全グリッドをスイープ"""
    model = SentenceTransformer(model_id, device="cuda")
    results = []

    for lang, articles in articles_dict.items():
        for size, overlap in tqdm(grid, desc=f"{model_id.split('/')[-1]} ({lang})"):
            chunks, article_ids = chunk_articles(articles, tokenizer, size, overlap)
            if len(chunks) < 100:
                print(f"  Skip size={size}, overlap={overlap}: only {len(chunks)} chunks")
                continue

            # チャンクサイズに応じてbatch_sizeを動的調整
            bs = adaptive_batch_size(size, batch_size)

            # Embedding
            texts = [f"{prefix}{c}" for c in chunks] if prefix else chunks
            emb = model.encode(texts, batch_size=bs,
                               normalize_embeddings=True, show_progress_bar=False)
            emb = emb.astype(np.float32)

            # イテレーションごとにGPUキャッシュを解放
            torch.cuda.empty_cache()

            # 保存
            model_short = model_id.split("/")[-1].replace("-", "_").lower()
            fname = f"102_{model_short}_{size}_{overlap}_{lang}.npy"
            np.save(os.path.join(DATA_DIR, fname), emb)

            # 評価（CPU上のnumpy演算のみ）
            ret = compute_retrieval_metrics(emb, article_ids, k=10)
            ads = compute_ads(emb, article_ids)

            # embeddingをメモリから解放
            del emb, texts, chunks
            gc.collect()

            results.append({
                "model": model_id.split("/")[-1],
                "lang": lang,
                "chunk_size": size,
                "overlap": overlap,
                "n_chunks": len(article_ids),
                "n_articles": len(np.unique(article_ids)),
                **ret,
                **ads,
            })
            print(f"  size={size}, overlap={overlap}, {lang}: "
                  f"{len(article_ids)} chunks, bs={bs}, R@10={ret['Recall@10']:.4f}, "
                  f"VRAM={torch.cuda.memory_allocated()/1024**2:.0f}MB")

    del model
    clear_gpu()
    return results

### 3.1 multilingual-e5-base

In [ ]:
articles_dict = {"ja": articles_ja, "en": articles_en}

# E5の中間結果が既にあればスキップ
e5_csv = os.path.join(DATA_DIR, "102_sweep_results_e5.csv")
if os.path.exists(e5_csv):
    print(f"E5 results already exist: {e5_csv} — skipping sweep")
    results_e5 = pd.read_csv(e5_csv).to_dict("records")
else:
    results_e5 = run_sweep(
        "intfloat/multilingual-e5-base", "passage: ",
        tokenizer_e5, e5_grid, articles_dict, batch_size=64
    )
    pd.DataFrame(results_e5).to_csv(e5_csv, index=False)

print(f"E5 results: {len(results_e5)} rows")

### 3.2 Qwen3-Embedding-0.6B

In [6]:
# GPUキャッシュをクリア
clear_gpu()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1024**2:.0f} MB allocated, "
      f"{torch.cuda.memory_reserved()/1024**2:.0f} MB reserved")

VRAM after cleanup: 1070 MB allocated, 1116 MB reserved


In [ ]:
# Qwen3の中間結果が既にあればスキップ
qwen_csv = os.path.join(DATA_DIR, "102_sweep_results_qwen.csv")
if os.path.exists(qwen_csv):
    print(f"Qwen3 results already exist: {qwen_csv} — skipping sweep")
    results_qwen = pd.read_csv(qwen_csv).to_dict("records")
else:
    results_qwen = run_sweep(
        "Qwen/Qwen3-Embedding-0.6B", "",
        tokenizer_qwen, qwen_grid, articles_dict, batch_size=32
    )
    pd.DataFrame(results_qwen).to_csv(qwen_csv, index=False)

print(f"Qwen3 results: {len(results_qwen)} rows")

In [ ]:
# 中間CSVから読み直して結合（途中再開時も動作する）
df_e5 = pd.read_csv(os.path.join(DATA_DIR, "102_sweep_results_e5.csv"))
df_qwen = pd.read_csv(os.path.join(DATA_DIR, "102_sweep_results_qwen.csv"))
df = pd.concat([df_e5, df_qwen], ignore_index=True)
df.to_csv(os.path.join(DATA_DIR, "102_sweep_results.csv"), index=False)
print(df[["model", "lang", "chunk_size", "overlap", "n_chunks", "Recall@10", "MRR", "ADS (AUC)"]].to_string(index=False))

## 4. ヒートマップ可視化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, model_name in enumerate(df["model"].unique()):
    for row, lang in enumerate(["ja", "en"]):
        ax = axes[row, col]
        subset = df[(df["model"] == model_name) & (df["lang"] == lang)]
        if subset.empty:
            ax.set_title(f"{model_name} ({lang}) - No data")
            continue

        pivot = subset.pivot_table(
            values="Recall@10", index="overlap", columns="chunk_size"
        )

        # matplotlib imshow でヒートマップ描画
        im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto",
                       vmin=pivot.values.min() - 0.02, vmax=pivot.values.max() + 0.02)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns.astype(int))
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index.astype(int))

        # 値をセルに表示
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                ax.text(j, i, f"{pivot.values[i, j]:.3f}",
                        ha="center", va="center", fontsize=9)

        ax.set_title(f"{model_name} ({lang.upper()})")
        ax.set_xlabel("chunk_size (tokens)")
        ax.set_ylabel("overlap (tokens)")
        fig.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Recall@10: chunk_size x overlap", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "102_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. オーバーラップ効果の分析

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lang in zip(axes, ["ja", "en"]):
    for model_name in df["model"].unique():
        subset = df[(df["model"] == model_name) & (df["lang"] == lang)]
        for size in sorted(subset["chunk_size"].unique()):
            sub = subset[subset["chunk_size"] == size].sort_values("overlap")
            ax.plot(sub["overlap"], sub["Recall@10"],
                    marker="o", label=f"{model_name} size={size}")

    ax.set_xlabel("Overlap (tokens)")
    ax.set_ylabel("Recall@10")
    ax.set_title(f"Overlap Effect ({lang.upper()})")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "102_overlap_effect.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. まとめ

In [ ]:
# 各モデル×言語で最良の組み合わせ
print("=== Best configurations ===")
for model_name in df["model"].unique():
    for lang in ["ja", "en"]:
        subset = df[(df["model"] == model_name) & (df["lang"] == lang)]
        if subset.empty:
            continue
        best = subset.loc[subset["Recall@10"].idxmax()]
        print(f"\n{model_name} ({lang.upper()}):")
        print(f"  Best: size={int(best['chunk_size'])}, overlap={int(best['overlap'])}")
        print(f"  Recall@10={best['Recall@10']:.4f}, MRR={best['MRR']:.4f}, ADS={best['ADS (AUC)']:.4f}")
        print(f"  Chunks: {int(best['n_chunks'])}")

# オーバーラップの収穫逓減点
print("\n=== Overlap diminishing returns ===")
for model_name in df["model"].unique():
    for lang in ["ja", "en"]:
        subset = df[(df["model"] == model_name) & (df["lang"] == lang)]
        if subset.empty:
            continue
        # 各サイズで、overlap=0からの改善率
        for size in sorted(subset["chunk_size"].unique()):
            sub = subset[subset["chunk_size"] == size].sort_values("overlap")
            base = sub.iloc[0]["Recall@10"]
            for _, row in sub.iterrows():
                delta = row["Recall@10"] - base
                if row["overlap"] > 0:
                    print(f"  {model_name} ({lang}) size={size} overlap={int(row['overlap'])}: "
                          f"R@10 delta={delta:+.4f}")

## 7. 評価・考察

### 主要結果

| モデル | 言語 | 最良 chunk_size | 最良 overlap | Recall@10 | MRR | ADS |
|--------|------|----------------|-------------|-----------|-----|-----|
| E5-base | JA | 256 | 64 | 0.681 | 0.944 | 0.950 |
| E5-base | EN | 384 | 128 | 0.791 | 0.976 | 0.991 |
| Qwen3-0.6B | JA | 2048 | 256 | 0.808 | 0.888 | 0.990 |
| Qwen3-0.6B | EN | 2048 | 256 | 0.877 | 0.900 | 0.999 |

### オーバーラップの効果

1. **オーバーラップは一貫して Recall@10 を改善する**。全モデル・全言語・全 chunk_size で overlap=0 より overlap>0 が優位。
2. **収穫逓減が明確に存在する**。E5-base (EN, size=256) では overlap 0→32 で +4.7pp、32→64 で +1.9pp と逓減。Qwen3 (EN, size=512) でも 0→64 が +2.4pp、64→128 が +2.4pp、128 以降は未測定だが同傾向と推測。
3. **実用的な推奨オーバーラップ率は 20〜25%**。E5 では overlap/size ≈ 25%（256/64, 384/128）、Qwen3 では ≈ 12.5〜25% が最良域。チャンク数の増加（≒インデックスサイズ・レイテンシ増）とのトレードオフを考えると、**overlap = chunk_size × 0.2〜0.25** が妥当。

### chunk_size の効果

1. **E5-base は小さめのチャンクが有利** (256〜384)。max_position=512 のモデルなので 512 トークンでは末尾が切り捨てられ性能が頭打ち。
2. **Qwen3 は大きいチャンクほど高性能**。2048 トークンが全条件で最良。長コンテキスト対応モデルの利点が発揮されている。ただしチャンク数が激減（JA: 3,696、EN: 3,121）するため、粒度の細かい検索には不向き。
3. **JA は EN より全体的に Recall が低い**（E5: −0.11pp、Qwen3: −0.07pp）。日本語はトークン効率が低く、同一トークン数でカバーするテキスト量が少ないことが一因。

### ADS (Article Discrimination Score) の傾向

- chunk_size が大きいほど ADS が高い（同一記事内チャンクの類似度が上がる）。
- Qwen3 の IIG（Intra-Inter Gap）は E5 の 3〜5 倍（0.22〜0.46 vs 0.06〜0.10）。Qwen3 は記事内の意味的一貫性をよりよく捉えている。

### NB103 以降への示唆

- **E5-base**: chunk_size=256, overlap=64 を基準設定として採用。
- **Qwen3**: chunk_size=512, overlap=128（粒度重視）または 1024/128（品質重視）を候補。2048 は粒度が粗すぎるため境界認識チャンキング (NB103) での改善余地が大きい。
- 固定長チャンキングの限界として、文境界・段落境界を無視した分割が Recall を下げている可能性がある。NB103 の boundary-aware chunking で改善を検証する。